# SAM3 Exploration: Building the Sam3MedSeg wrapper

Goal: understand SAM3's structure so we can extract the image encoder and attach our own segmentation head for dense binary medical masks (BUSI/Kvasir).
- Model I/O we must match: image (B,3,256,256) → logits (B,1,256,256)
- SAM3 is natively detect+track (not dense seg) → we reuse the backbone, not the full path.

In [1]:
import torch, inspect
from sam3.model_builder import build_sam3_image_model

model = build_sam3_image_model().to("cuda").eval()
print("loaded:", type(model).__module__, type(model).__name__)
print("CUDA:", torch.cuda.get_device_name(0))

/storage/ice1/4/1/skang411/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/storage/ice1/4/1/skang411/envs/sam-medseg/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/storage/ice1/4/1/skang411/envs/sam-medseg/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


loaded: sam3.model.sam3_image Sam3Image
CUDA: NVIDIA A100-PCIE-40GB


## Top-level structure
Which submodule is the image encoder, and how big is each piece?

In [5]:
for name, child in model.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"{name:22s} {type(child).__name__:32s} {n/1e6:8.1f}M")

backbone               SAM3VLBackbone                      807.8M
geometry_encoder       SequenceGeometryEncoder               8.2M
transformer            TransformerWrapper                   21.0M
segmentation_head      UniversalSegmentationHead             2.3M
dot_prod_scoring       DotProductScoring                     1.2M


## Can we run the backbone on an image alone?

The backbone is SAM3VLBackbone (Vision-Language), it may expect text too.
Test feeding just a dummy image, and inspect the output shape (our feature map).

In [3]:
print("forward signature:", inspect.signature(model.backbone.forward), "\n")

x = torch.randn(1, 3, 256, 256, device="cuda")

def shp(o):
    if torch.is_tensor(o): return tuple(o.shape)
    if isinstance(o, (list, tuple)): return [shp(i) for i in o]
    if isinstance(o, dict): return {k: shp(v) for k, v in o.items()}
    return type(o).__name__

with torch.no_grad():
    try:
        out = model.backbone(x)
        print("✅ backbone(image) works")
        print("output:", shp(out))
    except Exception as e:
        print("❌ backbone(image) failed:", type(e).__name__)
        print(str(e)[:400])

forward signature: (samples: torch.Tensor, captions: List[str], input_boxes: Optional[torch.Tensor] = None, additional_text: Optional[List[str]] = None) 

❌ backbone(image) failed: TypeError
SAM3VLBackbone.forward() missing 1 required positional argument: 'captions'


Task #1 (explore context) — largely done through this whole session, but let me consolidate what we know before questions. We have: the data contract, the empty stubs, and the SAM 3 structure. There's one gap: I want to see exactly how SAM3VLBackbone is built internally — whether there's a clean vision-only trunk we could tap (relevant to the first design question). Let me mark #1 in progress and grab that.

Why this matters for the design: the backbone forward requires captions, but internally it almost certainly has a separate image trunk (a ViT/Hiera that processes pixels) feeding a text-fusion stage. If we can see and reach that image trunk directly, we get a cleaner "SAM3 as pure vision backbone" option — which is one of the two approaches I'll propose. This tells us whether that path is even available.

In [6]:
# Inspect SAM3VLBackbone internals — is there a vision-only trunk we can tap?
bb = model.backbone
print("=== SAM3VLBackbone children ===")
for name, child in bb.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"{name:24s} {type(child).__name__:34s} {n/1e6:8.1f}M")

# attrs that look image/vision-only vs text
print("\n=== candidate vision vs text attrs ===")
for a in dir(bb):
    if a.startswith("_"): continue
    if any(k in a.lower() for k in ("vis","image","trunk","patch","pixel","text","lang","token","caption","embed")):
        print("  ", a)

=== SAM3VLBackbone children ===
vision_backbone          Sam3DualViTDetNeck                    454.0M
language_backbone        VETextEncoder                         353.7M

=== candidate vision vs text attrs ===
   act_ckpt_whole_language_backbone
   act_ckpt_whole_vision_backbone
   dump_patches
   forward_image
   forward_text
   language_backbone
   vision_backbone


In [7]:
# What does the vision-only path return? (shape of features we'll build the head on)
x = torch.randn(1, 3, 256, 256, device="cuda")

def shp(o):
    if torch.is_tensor(o): return tuple(o.shape)
    if isinstance(o,(list,tuple)): return [shp(i) for i in o]
    if isinstance(o,dict): return {k:shp(v) for k,v in o.items()}
    return type(o).__name__

import inspect
print("forward_image signature:", inspect.signature(model.backbone.forward_image))
print()
with torch.no_grad():
    try:
        feats = model.backbone.forward_image(x)
        print("✅ forward_image(x) works")
        print("output:", shp(feats))
    except Exception as e:
        print("❌ forward_image failed:", type(e).__name__, str(e)[:300])
        print("\ntrying vision_backbone directly...")
        try:
            feats = model.backbone.vision_backbone(x)
            print("✅ vision_backbone(x) works")
            print("output:", shp(feats))
        except Exception as e2:
            print("❌ also failed:", type(e2).__name__, str(e2)[:300])

forward_image signature: (samples: torch.Tensor)

❌ forward_image failed: RuntimeError mat1 and mat2 must have the same dtype, but got BFloat16 and Float

trying vision_backbone directly...
❌ also failed: RuntimeError mat1 and mat2 must have the same dtype, but got BFloat16 and Float


In [8]:
x = torch.randn(1, 3, 256, 256, device="cuda", dtype=torch.bfloat16)

def shp(o):
    if torch.is_tensor(o): return (tuple(o.shape), str(o.dtype))
    if isinstance(o,(list,tuple)): return [shp(i) for i in o]
    if isinstance(o,dict): return {k:shp(v) for k,v in o.items()}
    return type(o).__name__

with torch.no_grad():
    feats = model.backbone.forward_image(x)
print("✅ forward_image(bf16) works")
print("output:", shp(feats))

RuntimeError: Input type (CUDABFloat16Type) and weight type (torch.cuda.FloatTensor) should be the same

In [9]:
from collections import Counter
dtypes = Counter(p.dtype for p in model.backbone.vision_backbone.parameters())
print("vision_backbone param dtypes:", dict(dtypes))

# specifically the patch_embed conv that errored
pe = model.backbone.vision_backbone.trunk.patch_embed.proj
print("patch_embed.proj.weight dtype:", pe.weight.dtype)

# and check the whole model's dtype spread
all_dtypes = Counter(p.dtype for p in model.parameters())
print("whole model param dtypes:", dict(all_dtypes))

vision_backbone param dtypes: {torch.float32: 410}
patch_embed.proj.weight dtype: torch.float32
whole model param dtypes: {torch.float32: 1102}


In [10]:
x = torch.randn(1, 3, 256, 256, device="cuda", dtype=torch.float32)

def shp(o):
    if torch.is_tensor(o): return (tuple(o.shape), str(o.dtype))
    if isinstance(o,(list,tuple)): return [shp(i) for i in o]
    if isinstance(o,dict): return {k:shp(v) for k,v in o.items()}
    return type(o).__name__

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    feats = model.backbone.forward_image(x)

print("✅ forward_image under autocast works")
print("output:", shp(feats))

AssertionError: 

In [11]:
# What input size does SAM 3's ViT expect? Check the processor + freqs_cis geometry
import inspect

# 1. processor config often states the target size
proc_cfg = "/storage/ice1/4/1/skang411/checkpoints/sam3/processor_config.json"
import json
print("=== processor_config.json ===")
print(json.dumps(json.load(open(proc_cfg)), indent=2)[:1500])

# 2. the ViT's positional freq table shape hints at native grid size
vit = model.backbone.vision_backbone.trunk
for name, buf in vit.named_buffers():
    if "freqs" in name.lower() or "pos" in name.lower():
        print(f"{name}: {tuple(buf.shape)}")

=== processor_config.json ===
{
  "image_processor": {
    "crop_size": null,
    "data_format": "channels_first",
    "device": null,
    "disable_grouping": null,
    "do_center_crop": null,
    "do_convert_rgb": true,
    "do_normalize": true,
    "do_pad": null,
    "do_rescale": true,
    "do_resize": true,
    "image_mean": [
      0.5,
      0.5,
      0.5
    ],
    "image_processor_type": "Sam3ImageProcessorFast",
    "image_seq_length": null,
    "image_std": [
      0.5,
      0.5,
      0.5
    ],
    "input_data_format": null,
    "mask_size": {
      "height": 288,
      "width": 288
    },
    "pad_size": null,
    "processor_class": "Sam3VideoProcessor",
    "resample": 2,
    "rescale_factor": 0.00392156862745098,
    "return_tensors": null,
    "size": {
      "height": 1008,
      "width": 1008
    }
  },
  "processor_class": "Sam3VideoProcessor",
  "target_size": 1008,
  "video_processor": {
    "crop_size": null,
    "data_format": "channels_first",
    "default_to

In [12]:
import torch
torch.cuda.reset_peak_memory_stats()
x = torch.randn(1, 3, 1008, 1008, device="cuda", dtype=torch.float32)

def shp(o):
    if torch.is_tensor(o): return (tuple(o.shape), str(o.dtype))
    if isinstance(o,(list,tuple)): return [shp(i) for i in o]
    if isinstance(o,dict): return {k:shp(v) for k,v in o.items()}
    return type(o).__name__

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    feats = model.backbone.forward_image(x)

print("✅ forward_image @ 1008 works")
print("output:", shp(feats))
print(f"peak VRAM (inference, batch=1): {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

✅ forward_image @ 1008 works
output: {'vision_features': ((1, 256, 72, 72), 'torch.bfloat16'), 'vision_pos_enc': [((1, 256, 288, 288), 'torch.bfloat16'), ((1, 256, 144, 144), 'torch.bfloat16'), ((1, 256, 72, 72), 'torch.bfloat16')], 'backbone_fpn': [((1, 256, 288, 288), 'torch.bfloat16'), ((1, 256, 144, 144), 'torch.bfloat16'), ((1, 256, 72, 72), 'torch.bfloat16')], 'sam2_backbone_out': 'NoneType'}
peak VRAM (inference, batch=1): 4.41 GB
